# 01 - QLoRA fine-tune MedGemma-4B-it on MedMCQA, with a clean before/after evaluation
Runtime: **Colab T4** or **Kaggle T4** (Unsloth uses one GPU, so Kaggle's 2nd T4 does not help). ~40-60 min total.

**Splits (explicit, fixed seed):**
| split | source | n | used for |
|---|---|---|---|
| train | MedMCQA `train`, single-answer, shuffled seed 42, first N_TRAIN | 3000 | LoRA training |
| dev | MedMCQA `train`, next N_DEV after train (disjoint) | 300 | loss monitoring / any hyper-param choice |
| test | MedMCQA `validation`, single-answer, shuffled seed 42, first N_TEST, de-duplicated against train | 500 | reported once, base vs fine-tuned |

MedMCQA's official `test` split has no public labels, so `validation` serves as test. Hyper-parameters are chosen on dev only.

**Metric:** letter-logit accuracy - after the prompt we append `Answer:` and compare the model's next-token
log-probability for A/B/C/D. This removes the format-compliance confound (a base model that answers in prose
is not counted wrong just for format). Generation-based accuracy is reported separately as a secondary number.


In [ ]:
!pip install -q unsloth datasets scipy
import os
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"  # we need logits at eval time


In [ ]:
from huggingface_hub import login
login()  # MedGemma is under the Health AI Developer Foundations terms: accept them on the google/medgemma-4b-it page first


In [ ]:
MODEL = "unsloth/medgemma-4b-it-unsloth-bnb-4bit"   # 4-bit mirror of google/medgemma-4b-it
N_TRAIN, N_DEV, N_TEST = 3000, 300, 500
SEED, MAX_SEQ, R, LR, EPOCHS = 42, 1024, 16, 2e-4, 1
OUT = "outputs/medgemma-medmcqa-lora"
LETTERS = "ABCD"


In [ ]:
from datasets import load_dataset

def single(ds):
    return ds.filter(lambda r: r["choice_type"] == "single" and r["cop"] in (0, 1, 2, 3))

tr = single(load_dataset("openlifescienceai/medmcqa", split="train")).shuffle(seed=SEED)
train_ds = tr.select(range(N_TRAIN))
dev_ds = tr.select(range(N_TRAIN, N_TRAIN + N_DEV))

norm = lambda q: " ".join(q.lower().split())
train_q = {norm(q) for q in train_ds["question"]}
va = single(load_dataset("openlifescienceai/medmcqa", split="validation")).shuffle(seed=SEED)
va = va.filter(lambda r: norm(r["question"]) not in train_q)
test_ds = va.select(range(N_TEST))
print(len(train_ds), len(dev_ds), len(test_ds))


In [ ]:
def mcq_prompt(r):
    return (f"Answer the following medical multiple-choice question.\n\nQuestion: {r['question']}\n"
            f"A. {r['opa']}\nB. {r['opb']}\nC. {r['opc']}\nD. {r['opd']}\n\n"
            "Respond with 'Answer: <letter>' followed by a brief explanation.")

def target(r):
    exp = " ".join((r.get("exp") or "").split())[:400]
    return f"Answer: {LETTERS[r['cop']]}" + (f"\nExplanation: {exp}" if exp else "")


In [ ]:
import numpy as np, torch
from tqdm.auto import tqdm
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(MODEL, max_seq_length=MAX_SEQ, load_in_4bit=True, full_finetuning=False)
text_tok = getattr(tokenizer, "tokenizer", tokenizer)

def letter_ids(L):
    ids = set()
    for v in (L, " " + L):
        e = text_tok.encode(v, add_special_tokens=False)
        if len(e) == 1:
            ids.add(e[0])
    assert ids, L
    return sorted(ids)
CAND = {L: letter_ids(L) for L in LETTERS}
print(CAND)

def chat(r, with_answer=False):
    conv = [{"role": "user", "content": [{"type": "text", "text": mcq_prompt(r)}]}]
    if with_answer:
        conv.append({"role": "assistant", "content": [{"type": "text", "text": target(r)}]})
        return tokenizer.apply_chat_template(conv, tokenize=False)
    return tokenizer.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)

@torch.inference_mode()
def score_mcq(rows, desc):
    FastModel.for_inference(model)
    preds = []
    for r in tqdm(rows, desc=desc):
        enc = tokenizer(text=chat(r) + "Answer:", add_special_tokens=False, return_tensors="pt").to(model.device)
        lp = torch.log_softmax(model(**enc).logits[0, -1].float(), -1)
        preds.append(int(np.argmax([torch.logsumexp(lp[CAND[L]], 0).item() for L in LETTERS])))
    gold = np.array([r["cop"] for r in rows])
    return (np.array(preds) == gold), preds


## Baseline (no adapter attached yet)


In [ ]:
base_dev, _ = score_mcq(dev_ds, "base/dev")
base_test, base_pred = score_mcq(test_ds, "base/test")
print(f"BASE  dev {base_dev.mean():.3f}  test {base_test.mean():.3f}")


## QLoRA (language layers only - vision tower stays frozen so the VLM role is unchanged)


In [ ]:
model = FastModel.get_peft_model(
    model, finetune_vision_layers=False, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=R, lora_alpha=R, lora_dropout=0, bias="none", random_state=SEED)

train_txt = train_ds.map(lambda r: {"text": chat(r, True).removeprefix("<bos>")}, remove_columns=train_ds.column_names)
dev_txt = dev_ds.map(lambda r: {"text": chat(r, True).removeprefix("<bos>")}, remove_columns=dev_ds.column_names)
print(train_txt[0]["text"][:600])


In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth import is_bf16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model, tokenizer=text_tok, train_dataset=train_txt, eval_dataset=dev_txt,
    args=SFTConfig(
        dataset_text_field="text", per_device_train_batch_size=4, gradient_accumulation_steps=4,
        per_device_eval_batch_size=4, num_train_epochs=EPOCHS, learning_rate=LR, warmup_ratio=0.03,
        lr_scheduler_type="cosine", optim="adamw_8bit", weight_decay=0.01, logging_steps=10,
        eval_strategy="steps", eval_steps=50, fp16=not is_bf16_supported(), bf16=is_bf16_supported(),
        seed=SEED, report_to="none", output_dir="outputs/run"))
trainer = train_on_responses_only(trainer, instruction_part="<start_of_turn>user\n",
                                  response_part="<start_of_turn>model\n")
stats = trainer.train()


In [ ]:
model.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)


## Fine-tuned evaluation + paired statistics


In [ ]:
ft_dev, _ = score_mcq(dev_ds, "ft/dev")
ft_test, ft_pred = score_mcq(test_ds, "ft/test")

from scipy.stats import binomtest
b = int((base_test & ~ft_test).sum())   # base right, ft wrong
c = int((~base_test & ft_test).sum())   # base wrong, ft right
p = binomtest(min(b, c), b + c, 0.5).pvalue if b + c else 1.0
rng = np.random.default_rng(SEED)
boot = [ft_test[i].mean() - base_test[i].mean() for i in (rng.integers(0, N_TEST, N_TEST) for _ in range(5000))]
lo, hi = np.percentile(boot, [2.5, 97.5])

res = {"model": MODEL, "splits": {"train": N_TRAIN, "dev": N_DEV, "test": N_TEST, "seed": SEED,
                                  "test_source": "medmcqa/validation (dedup vs train)"},
       "base": {"dev": float(base_dev.mean()), "test": float(base_test.mean())},
       "finetuned": {"dev": float(ft_dev.mean()), "test": float(ft_test.mean())},
       "delta_test": float(ft_test.mean() - base_test.mean()), "delta_test_95ci": [float(lo), float(hi)],
       "mcnemar": {"base_only_correct": b, "ft_only_correct": c, "p_exact": float(p)}}
import json; os.makedirs("results", exist_ok=True)
json.dump(res, open("results/mcq_results.json", "w"), indent=2)
print(json.dumps(res, indent=2))


## Sanity checks
1. With the adapter disabled the model must reproduce the baseline (proves the VLM role is untouched).
2. Generation-based accuracy + format compliance on 100 test items (secondary metric).


In [ ]:
with model.disable_adapter():
    chk, _ = score_mcq(test_ds.select(range(100)), "adapter-off check")
print("adapter-off acc on first 100:", chk.mean(), "| baseline on same 100:", base_test[:100].mean())


In [ ]:
import re
@torch.inference_mode()
def gen_acc(rows, adapter=True):
    FastModel.for_inference(model)
    ok = parsed = 0
    for r in tqdm(rows):
        enc = tokenizer(text=chat(r), add_special_tokens=False, return_tensors="pt").to(model.device)
        def go():
            return model.generate(**enc, max_new_tokens=12, do_sample=False)
        if adapter:
            out = go()
        else:
            with model.disable_adapter():
                out = go()
        txt = text_tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        m = re.search(r"Answer:\s*\**\(?([ABCD])", txt) or re.match(r"\s*\**\(?([ABCD])\b", txt)
        if m:
            parsed += 1
            ok += LETTERS.index(m.group(1)) == r["cop"]
    return ok / len(rows), parsed / len(rows)

sub = test_ds.select(range(100))
print("generation acc / parse-rate  base:", gen_acc(sub, False), " ft:", gen_acc(sub, True))


In [ ]:
# Download the adapter (or push: model.push_to_hub("<you>/medgemma-medmcqa-lora", private=True))
!zip -r medgemma-medmcqa-lora.zip outputs/medgemma-medmcqa-lora results
